<a href="https://colab.research.google.com/github/Anast2007/Assignment-AIML/blob/main/CarPrices.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
path = kagglehub.dataset_download("doaaalsenani/usa-cers-dataset")

### Data Preprocessing for Machine Learning

First, we need to prepare the data for training our machine learning models. This involves selecting features, handling categorical variables, and splitting the dataset into training and testing sets.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Define features (X) and target (y)
X = df.drop('price', axis=1)
y = df['price']

# Identify categorical and numerical columns
categorical_features = ['brand', 'model', 'condition', 'state', 'color', 'country']
numerical_features = ['year', 'mileage'] # Assuming these are the main numerical features from the dataset description or common car datasets.

# Create a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

### Model Training and Evaluation

Now we will train the specified models (Linear Regression, Decision Tree Regressor, and Random Forest Regressor) and evaluate their performance using Mean Absolute Error (MAE), Mean Squared Error (MSE), and R-squared score.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Function to train and evaluate a model
def train_evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    print(f"\n--- {model_name} ---")

    # Create a pipeline with preprocessing and model
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('regressor', model)])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"Mean Absolute Error (MAE): {mae:.2f}")
    print(f"Mean Squared Error (MSE): {mse:.2f}")
    print(f"R-squared (R2): {r2:.2f}")
    return mae, mse, r2

# 1. Linear Regression
linear_reg_model = LinearRegression()
linear_reg_metrics = train_evaluate_model(linear_reg_model, X_train, y_train, X_test, y_test, "Linear Regression")

# 2. Decision Tree Regressor
decision_tree_model = DecisionTreeRegressor(random_state=42)
decision_tree_metrics = train_evaluate_model(decision_tree_model, X_train, y_train, X_test, y_test, "Decision Tree Regressor")

# 3. Random Forest Regressor
random_forest_model = RandomForestRegressor(random_state=42, n_estimators=100)
random_forest_metrics = train_evaluate_model(random_forest_model, X_train, y_train, X_test, y_test, "Random Forest Regressor")

#### A Note on Logistic Regression

YouYou requested Logistic Regression. However, Logistic Regression is a classification algorithm used for predicting discrete, categorical outcomes (e.g., 'yes' or 'no', 'spam' or 'not spam', different car types). Since we are predicting car prices, which is a continuous numerical value, regression algorithms (like Linear Regression, Decision Tree Regressor, Random Forest Regressor) are more appropriate.

If you intended to perform a classification task (e.g., predict if a car is 'expensive' vs. 'affordable' based on a price threshold), we would first need to create a categorical target variable from the 'price' column. Please let me know if you would like to explore such a classification task.

In [ ]:
print("Outlier rows for 'land':")
if 'land' in outliers_data:
    display(df.loc[outliers_data['land'].index])
else:
    print("No outliers found for 'land'.")

print("\nOutlier rows for 'mercedes-benz':")
if 'mercedes-benz' in outliers_data:
    display(df.loc[outliers_data['mercedes-benz'].index])
else:
    print("No outliers found for 'mercedes-benz'.")

These rows provide more context about the outlier vehicles, including their models, years, mileage, and other attributes, which can help in understanding why their prices are significantly different.

In [ ]:
import numpy as np

outliers_data = {}

for brand in top_5_brands:
    brand_prices = df_top_brands[df_top_brands['brand'] == brand]['price']

    # Calculate Q1, Q3, and IQR
    Q1 = brand_prices.quantile(0.25)
    Q3 = brand_prices.quantile(0.75)
    IQR = Q3 - Q1

    # Define outlier bounds
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Identify outliers
    outliers = brand_prices[(brand_prices < lower_bound) | (brand_prices > upper_bound)]

    if not outliers.empty:
        outliers_data[brand] = outliers

# Display the identified outliers
if outliers_data:
    for brand, outliers in outliers_data.items():
        print(f"\nOutliers for {brand} (count: {len(outliers)}):")
        display(outliers)
else:
    print("No significant outliers identified based on the 1.5*IQR rule for the top 5 brands.")

The output above lists the price outliers for each of the top 5 car brands, if any were found based on the 1.5*IQR rule. This can help identify unusually high or low-priced vehicles within these brands, which might be due to rare models, special editions, or data entry errors.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get the top 5 car brands based on average price
top_5_brands = average_price_by_brand.head(5).index

# Filter the DataFrame to include only these top 5 brands
df_top_brands = df[df['brand'].isin(top_5_brands)]

# Create a box plot to show the price distribution for the top 5 brands
plt.figure(figsize=(12, 7))
sns.boxplot(x='brand', y='price', data=df_top_brands, order=top_5_brands, palette='viridis', hue='brand', legend=False)
plt.title('Price Distribution for Top 5 Car Brands')
plt.xlabel('Car Brand')
plt.ylabel('Price (USD)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate average price per car brand
average_price_by_brand = df.groupby('brand')['price'].mean().sort_values(ascending=False)

# Create the bar chart
plt.figure(figsize=(12, 7))
sns.barplot(x=average_price_by_brand.index, y=average_price_by_brand.values, palette='viridis')
plt.title('Average Car Price by Brand')
plt.xlabel('Car Brand')
plt.ylabel('Average Price (USD)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
df.info()

In [ ]:
import pandas as pd

# Define the thresholds and corresponding labels for car price categories
bins = [0, 15000, 35000, float('inf')] # 0 to 15k, 15k to 35k, 35k to infinity
labels = ['Affordable', 'Mid-range', 'Luxury']

# Create the new categorical column 'price_category'
df['price_category'] = pd.cut(df['price'], bins=bins, labels=labels, right=True)

# Display the first few rows with the new column
display(df[['price', 'price_category']].head())

# Display the counts of each new category to check distribution
print("\nDistribution of Car Price Categories:")
display(df['price_category'].value_counts())

In [ ]:
import pandas as pd
import os

file_path = os.path.join(path, 'USA_cars_datasets.csv')
df = pd.read_csv(file_path)

# Display the first 5 rows of the DataFrame
display(df.head())

In [ ]:
import os

# List the contents of the downloaded directory
print(f"Contents of {path}:")
for root, dirs, files in os.walk(path):
    for name in files:
        print(os.path.join(root, name))
    for name in dirs:
        print(os.path.join(root, name))